<a href="https://colab.research.google.com/github/vaibhavbazaria/HMM_PROJECT-/blob/main/HMM_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0],
                                  [0.0, 0.9, 0.1, 0.0, 0.0],
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]])
emission_nuc_codes = {'A': 0,
                      'C': 1,
                      'G': 2,
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00],
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]])

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

In [27]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [28]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)

-43.89740030179307


-43.89740030179307

In [29]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)

-43.45111319916465


In [30]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)

-43.944833355027704


In [31]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)

-42.58225552052512


In [32]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)

-41.21967768602254


In [33]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)

-41.713397841885595


In [34]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)


-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides.

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .]
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .]
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

In [35]:
num_states = len(states)
seq_len = len(query_sequence)

viterbi_value_matrix = np.full((num_states, seq_len), -np.inf) # Initialize with negative infinity for log probabilities
viterbi_trace_matrix = np.full((num_states, seq_len), -1, dtype=int)

# Initialize the first column (for the first observation)
# The 's' (start) state transitions to 'E' with probability 1.0.
# The 's' and 'e' states do not emit. Emissions start from E.

# Calculate for state 'E' (index 1) for the first nucleotide
first_nucleotide_code = emission_nuc_codes[query_sequence[0]]
log_initial_prob_E = math.log(state_transition_prob[states['s'], states['E']]) + math.log(emission_probs[states['E'], first_nucleotide_code])
viterbi_value_matrix[states['E'], 0] = log_initial_prob_E
viterbi_trace_matrix[states['E'], 0] = states['s'] # Trace back to the start state 's'

# Display initial matrices (optional, for debugging)
# print("Initial Viterbi Value Matrix:")
# print(viterbi_value_matrix)
# print("Initial Viterbi Trace Matrix:")
# print(viterbi_trace_matrix)

### Implementation of Viterbi Algorithm
Write a function `calculate_prob_for_a_node()` that populate a single cell in the matrix. The function will return two values:
1. the maximum value, for example, look at the 2nd row, 2nd column in the matrix: `max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)`. If the probability for `s-E-C-E-T` is highest (lets say X), then the function should return `X`

**AND**

2. The index of that maximum value described in the first point: so index of X is `1` (recall that Python works on the 0-based index system)

- Populate `viterbi_value_matrix` with `X` for row 2 and col 2

- Populate `viterbi_trace_matrix` with `1` for row 2 and col 2

In [38]:
def calculate_prob_for_a_node(current_state_idx, time_step, viterbi_value_matrix, state_transition_prob, emission_probs, query_sequence, emission_nuc_codes):
    """
    Calculates the maximum log probability and the corresponding previous state index
    for a specific cell (state, time_step) in the Viterbi matrix.
    """
    nuc_code = emission_nuc_codes[query_sequence[time_step]]
    log_emission = math.log(emission_probs[current_state_idx, nuc_code]) if emission_probs[current_state_idx, nuc_code] > 0 else -np.inf

    max_val = -np.inf
    best_prev_state = -1

    # Iterate through all possible previous states
    for prev_state_idx in range(num_states):
        prev_prob = viterbi_value_matrix[prev_state_idx, time_step - 1]
        trans_prob = state_transition_prob[prev_state_idx, current_state_idx]

        if prev_prob > -np.inf and trans_prob > 0:
            current_prob = prev_prob + math.log(trans_prob) + log_emission
            if current_prob > max_val:
                max_val = current_prob
                best_prev_state = prev_state_idx

    return max_val, best_prev_state

In [36]:
# Write for loops to iterate over the whole Viterbi Value matrix.
# Each time, call the function

In [39]:
# Iterate through the sequence (starting from the second nucleotide)
for t in range(1, seq_len):
    for s_idx in range(num_states):
        # Skip start and end states if they don't emit (s=0, e=4)
        if s_idx == 0 or s_idx == 4:
            continue

        val, trace = calculate_prob_for_a_node(s_idx, t, viterbi_value_matrix, state_transition_prob, emission_probs, query_sequence, emission_nuc_codes)
        viterbi_value_matrix[s_idx, t] = val
        viterbi_trace_matrix[s_idx, t] = trace

print("Viterbi Value Matrix populated.")

Viterbi Value Matrix populated.


In [37]:
# Write a function to trace the state path that gave the maximum probability.
# This will be the final result.


# HINT: You should first find the maximum value in the last column of `viterbi_value_matrix`,
# because that is the one with the largest probability.
# The index of that value is the state of the last nucleotide.

In [40]:
def backtrack_viterbi(viterbi_value_matrix, viterbi_trace_matrix, id2state):
    # 1. Find the best state at the last nucleotide
    last_col = viterbi_value_matrix[:, -1]
    best_last_state = np.argmax(last_col)
    max_log_prob = last_col[best_last_state]

    # 2. Backtrack to find the path
    path = []
    curr_state = best_last_state
    for t in range(seq_len - 1, -1, -1):
        path.append(id2state[curr_state])
        curr_state = viterbi_trace_matrix[curr_state, t]

    return "".join(reversed(path)), max_log_prob

final_path, final_log_prob = backtrack_viterbi(viterbi_value_matrix, viterbi_trace_matrix, id2state)
print(f"Most probable path: {final_path}")
print(f"Log probability: {final_log_prob}")

Most probable path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Log probability: -38.677666280562796
